# Artificial Neural Network (ANN) Classification Pipeline

### Overview
This notebook introduces a custom Artificial Neural Network (ANN) architecture to serve as a comparative Deep Learning model for our classification project. While previous pipelines established classical Machine Learning baselines and advanced Computer Vision networks, this notebook explores how a dense Multilayer Perceptron (MLP) handles the extracted feature representations.

### Pipeline Objectives
1. **Data Preprocessing & Flattening:** Ingesting the cleaned dataset and transforming the features into the optimal 1D tensor format required by the dense feedforward network.
2. **Architecture Construction:** Designing a robust sequence of hidden layers, utilizing dropout mechanisms to prevent memorization, and applying the correct activation functions.
3. **Training & Optimization:** Compiling the network with an appropriate loss function and executing the training loop with Early Stopping safeguards.
4. **Performance Evaluation:** Generating interactive visualization dashboards to compare the ANN's validation accuracy and recall metrics against our established benchmarks.

## 1. Environment Initialization & Core Libraries
Setting up the Deep Learning workspace by importing the necessary data manipulation, machine learning, and visualization frameworks. 

### Key Dependencies:
* **TensorFlow / Keras:** The primary framework used to architect the Artificial Neural Network (`Sequential`). We import `Dense` layers for the core learning capacity, alongside `Dropout` and `BatchNormalization` to regularize the network and prevent overfitting on the clinical data.
* **Scikit-Learn:** Utilized for securing a pure Train/Validation split and generating critical medical diagnostic metrics (ROC, AUC, Precision-Recall).
* **Plotly:** Powers the interactive, dynamic dashboards used to visualize network performance during and after training.

In [2]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import roc_curve, auc, precision_recall_curve

import plotly.express as px

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

In [3]:
df = pd.read_csv('../../data/Processed/Alzheimer_final.csv')
df.head()

,ID,Age,Gender,Educ,MMSE,Target,eTIV,nWBV
0,011_S_0003,0.717371,1,4.0,0.081752,1.0,0.014743,0.223724
1,022_S_0004,-0.854411,1,0.0,0.836511,1.0,0.014743,0.223724
2,011_S_0005,-0.148249,1,3.0,1.052156,0.0,0.014743,0.223724
3,100_S_0006,0.614863,0,2.0,0.620865,1.0,0.014743,0.223724
4,011_S_0010,-0.125469,0,1.0,0.513043,1.0,0.014743,0.223724


In [4]:
X = df.drop(columns=['ID', 'Target'])
y = df['Target'].astype(int)

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

In [6]:
ann_model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    BatchNormalization(),
    Dropout(0.3),

    Dense(32, activation='relu'),
    BatchNormalization(),
    Dropout(0.2),

    Dense(16, activation='relu'),
    Dropout(0.1),

    Dense(1, activation='sigmoid')
])

c:\Users\zyado\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


## 3. Network Compilation
Before the network can learn, it must be compiled with a specific optimization strategy and loss function. 
* **Optimizer (`Adam`):** Selected for its adaptive learning rate, making it highly efficient at finding the global minimum in the loss landscape.
* **Loss Function (`binary_crossentropy`):** The standard metric for binary classification tasks, penalizing the network heavily for confident but incorrect predictions.
* **Metrics:** Tracking Accuracy, Precision, Recall, and AUC dynamically during training to ensure the model prioritizes diagnostic sensitivity.

In [9]:
ann_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.AUC(name='auc')
    ]
)

ann_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,457 (13.50 KB)

 Trainable params: 3,265 (12.75 KB)

 Non-trainable params: 192 (768.00 B)

## 4. Model Training Phase
Executing the training loop over the tabular dataset. 
To prevent the model from memorizing the training data (overfitting), an `EarlyStopping` callback is deployed. It monitors the `val_loss`; if the validation loss fails to improve for 10 consecutive epochs, training is automatically halted, and the network restores its best-performing weights.

In [10]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True
)

history = ann_model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=150,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/150
60/60 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.4823 - auc: 0.5303 - loss: 0.7890 - precision: 0.4348 - recall: 0.7009 - val_accuracy: 0.6173 - val_auc: 0.6575 - val_loss: 0.6696 - val_precision: 0.6415 - val_recall: 0.3223
Epoch 2/150
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5632 - auc: 0.5871 - loss: 0.6879 - precision: 0.4895 - recall: 0.4895 - val_accuracy: 0.6321 - val_auc: 0.6643 - val_loss: 0.6591 - val_precision: 0.7284 - val_recall: 0.2796
Epoch 3/150
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6050 - auc: 0.6188 - loss: 0.6679 - precision: 0.5497 - recall: 0.4240 - val_accuracy: 0.6279 - val_auc: 0.6682 - val_loss: 0.6521 - val_precision: 0.6966 - val_recall: 0.2938
Epoch 4/150
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6050 - auc: 0.6326 - loss: 0.6544 - precision: 0.5527 - recall: 0.4017 - val_accuracy: 0.6237 - val_auc: 0.6739 - val_loss: 0.6462 - val_precision: 0.7143 - val_recall: 0.2607
Epoch 5/150
60/60 ━━━━━━━━━━━━━━━━━━

## 5. Final Evaluation Metrics
Testing the fully trained ANN on the completely untouched 30% holdout validation set (`X_test`). This step generates the final objective metrics (Loss, Accuracy, Precision, Recall, and AUC) to compare against our classical Support Vector Machine (SVM) baselines.

In [11]:
test_loss, test_accuracy, test_precision, test_recall, test_auc = ann_model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)
print("Test Precision:", test_precision)
print("Test Recall:", test_recall)
print("Test AUC:", test_auc)

Test Loss: 0.5895809531211853
Test Accuracy: 0.6577908992767334
Test Precision: 0.6952789425849915
Test Recall: 0.36986300349235535
Test AUC: 0.7156127691268921


In [12]:
y_prob = ann_model.predict(X_test).ravel()
y_pred = (y_prob >= 0.50).astype(int)

32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step


In [13]:
print("--- ANN Baseline Results ---\n")
print(classification_report(
    y_test,
    y_pred,
    target_names=['Healthy (0)', 'Early-Stage (1)']
))

--- ANN Baseline Results ---

                 precision    recall  f1-score   support

    Healthy (0)       0.65      0.88      0.74       576
Early-Stage (1)       0.70      0.37      0.48       438

       accuracy                           0.66      1014
      macro avg       0.67      0.62      0.61      1014
   weighted avg       0.67      0.66      0.63      1014



## 6. Performance Visualization Dashboard
Generating an interactive Plotly evaluation suite to analyze the model's diagnostic behavior:
* **Training vs. Validation Curves:** Confirms that the model generalized well without overfitting (the curves should ideally converge).
* **Confusion Matrix:** Visualizes the exact number of True Positives (correctly diagnosed Alzheimer's) versus False Negatives (missed diagnoses).
* **ROC & Precision-Recall Curves:** Demonstrates the network's classification threshold robustness.

In [14]:
cm = confusion_matrix(y_test, y_pred)

fig_cm = px.imshow(
    cm,
    text_auto=True,
    color_continuous_scale='Blues',
    labels=dict(
        x="Predicted Diagnosis",
        y="Actual Diagnosis",
        color="Patients"
    ),
    x=['Healthy (0)', 'Early-Stage (1)'],
    y=['Healthy (0)', 'Early-Stage (1)'],
    title="Interactive Confusion Matrix"
)

fig_cm.update_layout(
    title_x=0.5,
    width=600,
    height=600
)

fig_cm.show()

In [15]:
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

fig_roc = px.area(
    x=fpr,
    y=tpr,
    title=f'Interactive ROC Curve (AUC = {roc_auc:.4f})',
    labels=dict(
        x='False Positive Rate (1 - Specificity)',
        y='True Positive Rate (Sensitivity/Recall)'
    ),
    width=700,
    height=600,
    color_discrete_sequence=['#1f77b4']
)

fig_roc.add_shape(
    type='line',
    line=dict(dash='dash', color='gray'),
    x0=0,
    x1=1,
    y0=0,
    y1=1
)

fig_roc.update_layout(title_x=0.5)
fig_roc.show()

In [16]:
custom_threshold = 0.40

y_pred_medical = (y_prob >= custom_threshold).astype(int)

print(f"--- Results with {custom_threshold*100}% Threshold ---")
print(classification_report(
    y_test,
    y_pred_medical,
    target_names=['Healthy (0)', 'Early-Stage (1)']
))

--- Results with 40.0% Threshold ---
                 precision    recall  f1-score   support

    Healthy (0)       0.74      0.53      0.62       576
Early-Stage (1)       0.55      0.75      0.63       438

       accuracy                           0.63      1014
      macro avg       0.64      0.64      0.63      1014
   weighted avg       0.66      0.63      0.63      1014



In [17]:
cm_40 = confusion_matrix(y_test, y_pred_medical)

fig_cm = px.imshow(
    cm_40,
    text_auto=True,
    color_continuous_scale='blues',
    labels=dict(
        x="Predicted Diagnosis",
        y="Actual Diagnosis",
        color="Patients"
    ),
    x=['Healthy (0)', 'Early-Stage (1)'],
    y=['Healthy (0)', 'Early-Stage (1)'],
    title="Confusion Matrix at 40% Threshold"
)

fig_cm.update_layout(
    title_x=0.5,
    width=650,
    height=600
)

fig_cm.show()

In [18]:
precisions, recalls, thresholds = precision_recall_curve(y_test, y_prob)

tradeoff_df = pd.DataFrame({
    'Threshold': thresholds,
    'Precision': precisions[:-1],
    'Recall': recalls[:-1]
})

tradeoff_melted = tradeoff_df.melt(
    id_vars='Threshold',
    value_vars=['Precision', 'Recall'],
    var_name='Metric',
    value_name='Score'
)

In [19]:
fig_tradeoff = px.line(
    tradeoff_melted,
    x='Threshold',
    y='Score',
    color='Metric',
    title="Precision-Recall Trade-Off Analysis",
    labels={'Score': 'Percentage (0 to 1)'},
    color_discrete_sequence=['#ff7f0e', '#1f77b4']
)

fig_tradeoff.add_vline(
    x=0.40,
    line_width=3,
    line_dash="dash",
    line_color="red",
    annotation_text="Our 40% Medical Target",
    annotation_position="top right"
)

fig_tradeoff.update_layout(
    title_x=0.5,
    width=800,
    height=500
)

fig_tradeoff.show()

In [20]:
history_df = pd.DataFrame(history.history)
history_df['Epoch'] = range(1, len(history_df) + 1)

In [21]:
fig_acc = px.line(
    history_df,
    x='Epoch',
    y=['accuracy', 'val_accuracy'],
    title='ANN Training vs Validation Accuracy',
    labels={
        'value': 'Accuracy',
        'variable': 'Metric'
    }
)

fig_acc.update_layout(
    title_x=0.5,
    width=800,
    height=500
)

fig_acc.show()

In [22]:
fig_loss = px.line(
    history_df,
    x='Epoch',
    y=['loss', 'val_loss'],
    title='ANN Training vs Validation Loss',
    labels={
        'value': 'Loss',
        'variable': 'Metric'
    }
)

fig_loss.update_layout(
    title_x=0.5,
    width=800,
    height=500
)

fig_loss.show()

In [23]:
fig_auc = px.line(
    history_df,
    x='Epoch',
    y=['auc', 'val_auc'],
    title='ANN Training vs Validation AUC',
    labels={
        'value': 'AUC',
        'variable': 'Metric'
    }
)

fig_auc.update_layout(
    title_x=0.5,
    width=800,
    height=500
)

fig_auc.show()

## Conclusion & Engineering Summary
This notebook successfully implements a custom Multi-Layer Perceptron (ANN) to classify early-stage Alzheimer's Disease from clinical tabular data. 

By applying strict data regularization (`BatchNormalization` and `Dropout`) alongside a shrinking feature-funnel architecture, the model avoided overfitting and demonstrated strong generalization on the holdout test set. These metrics will now serve as our primary Deep Learning baseline as we transition to analyzing complex spatial representations (MRI images) in the subsequent Convolutional Neural Network (CNN) and DenseNet architectures.